In [ ]:
from google.colab import files
import io
uploads=files.upload()

Saving whisper_data.zip to whisper_data.zip


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
zip_path = '/content/drive/MyDrive/audios.zip'


In [ ]:
import zipfile
import os

# 3. Extraire comme avant dans un dossier "audios"
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("audios")  # ← tu peux maintenant utiliser ce dossier !

# 4. Vérifier les fichiers extraits
print("Fichiers extraits :", os.listdir("audios")[:5])


Fichiers extraits : ['audios']


In [ ]:
audios = "audios"


In [ ]:
os.path.join(audios, "nom_du_fichier.wav")


'audios/nom_du_fichier.wav'

In [ ]:
import zipfile
import os

# Remplace "bambara.zip" par le nom exact du fichier zip que tu as téléversé
with zipfile.ZipFile("whisper_jeli.zip", 'r') as zip_ref:
    zip_ref.extractall("whisper_jeli")  # Dossier où les fichiers seront extraits

# Vérifie le contenu
whisper_jeli=os.listdir("whisper_jeli")


In [ ]:
import zipfile
import os

# Remplace "bambara.zip" par le nom exact du fichier zip que tu as téléversé
with zipfile.ZipFile("whisper_data.zip", 'r') as zip_ref:
    zip_ref.extractall("whisper_data")  # Dossier où les fichiers seront extraits

# Vérifie le contenu
whisper_data=os.listdir("whisper_data")


In [ ]:
audios = os.listdir("audios")


In [ ]:
from transformers.models.whisper.tokenization_whisper import TO_LANGUAGE_CODE

TO_LANGUAGE_CODE

{'english': 'en',
 'chinese': 'zh',
 'german': 'de',
 'spanish': 'es',
 'russian': 'ru',
 'korean': 'ko',
 'french': 'fr',
 'japanese': 'ja',
 'portuguese': 'pt',
 'turkish': 'tr',
 'polish': 'pl',
 'catalan': 'ca',
 'dutch': 'nl',
 'arabic': 'ar',
 'swedish': 'sv',
 'italian': 'it',
 'indonesian': 'id',
 'hindi': 'hi',
 'finnish': 'fi',
 'vietnamese': 'vi',
 'hebrew': 'he',
 'ukrainian': 'uk',
 'greek': 'el',
 'malay': 'ms',
 'czech': 'cs',
 'romanian': 'ro',
 'danish': 'da',
 'hungarian': 'hu',
 'tamil': 'ta',
 'norwegian': 'no',
 'thai': 'th',
 'urdu': 'ur',
 'croatian': 'hr',
 'bulgarian': 'bg',
 'lithuanian': 'lt',
 'latin': 'la',
 'maori': 'mi',
 'malayalam': 'ml',
 'welsh': 'cy',
 'slovak': 'sk',
 'telugu': 'te',
 'persian': 'fa',
 'latvian': 'lv',
 'bengali': 'bn',
 'serbian': 'sr',
 'azerbaijani': 'az',
 'slovenian': 'sl',
 'kannada': 'kn',
 'estonian': 'et',
 'macedonian': 'mk',
 'breton': 'br',
 'basque': 'eu',
 'icelandic': 'is',
 'armenian': 'hy',
 'nepali': 'ne',
 'mongol

In [ ]:
# on va utiliser le francais comme point de depart afin que le modele soit pas tout bete bete au depart
from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained(
    "openai/whisper-small", language="french", task="transcribe"
)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
import os
import json

# json_dir = "Whisper_data/Whisper_data"
audio_dir = "audios/audios"
json_dir = "whisper_data/whisper_data"

dataset_entries = []

for filename in os.listdir(json_dir):
    if filename.endswith(".json"):
        json_path = os.path.join(json_dir, filename)

        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)

            audio_filename = data["audio_filename"]
            audio_path = os.path.join(audio_dir, f"{audio_filename}.wav")

            for segment in data["transcription"]:
                # ✅ Vérifie que les champs nécessaires existent
                if all(key in segment for key in ["start", "end", "text", "translation"]):
                    entry = {
                        "audio_filepath": audio_path,
                        "start": segment["start"],
                        "end": segment["end"],
                        "text": segment["text"],
                        "translation": segment["translation"]
                    }
                    dataset_entries.append(entry)
                else:
                    print(f"⛔ Segment ignoré dans {filename} car il manque une clé.")

print(f"\n✅ Nombre total de segments extraits : {len(dataset_entries)}")
if dataset_entries:
    print("🔍 Exemple d'entrée :")
    print(dataset_entries[0])



✅ Nombre total de segments extraits : 187
🔍 Exemple d'entrée :
{'audio_filepath': 'audios/audios/Apprendre le Bambara Tranquillement au Lit  200 PhrasesLire ou Écouter ou Répéter  Zanga School_segment_911.wav', 'start': 0.0, 'end': 23.83, 'text': 'Tan in tila ka ɲɛ, nin sira in tila ka ɲɛ, nin ye mun fɛn ye?', 'translation': "Divisez correctement le dix, divisez correctement cette ligne, qu'est-ce que c'est ?"}


In [ ]:
import os
import json


json_path = "whisper_jeli/whisper_jeli/jeli-asr-rmai-train-manifest.json"
audio_dir = "audios/audios"
jeli_entries = []

with open(json_path, "r", encoding="utf-8") as f:
    for line in f:
        try:
            data = json.loads(line.strip())

            if all(key in data for key in ["audio_filepath", "text", "duration"]):
                audio_file = os.path.basename(data["audio_filepath"])  # extrait juste le nom du fichier
                corrected_path = os.path.join(audio_dir, audio_file)   # nouveau chemin

                entry = {
                    "audio_filepath": corrected_path,
                    "start": 0.0,  # ⚠️ on suppose que l'audio commence à 0.0 s
                    "end": data["duration"],
                    "text": data["text"],
                    "translation": ""
                }
                jeli_entries.append(entry)
        except json.JSONDecodeError as e:
            print(f"⛔ JSON invalide : {e}")

# Résumé
print(f"\n✅ Nombre total d'entrées Jeli : {len(jeli_entries)}")
print("🔍 Exemple :", jeli_entries[0] if jeli_entries else "aucune")



✅ Nombre total d'entrées Jeli : 1715
🔍 Exemple : {'audio_filepath': 'audios/audios/griots_r8-646950-653370.wav', 'start': 0.0, 'end': 6.42, 'text': "minun nana kita bolo fɛ o tun y'a ta kɛ woro feere de ye", 'translation': ''}


In [ ]:
import os

# On teste les 5 premiers fichiers (ou moins si tu en as moins)
print("\n🎧 Vérification des fichiers audio :")
for i, entry in enumerate(jeli_entries[:5]):
    path = entry["audio_filepath"]
    if os.path.exists(path):
        print(f"✅ Fichier trouvé : {path}")
    else:
        print(f"❌ Fichier manquant : {path}")



🎧 Vérification des fichiers audio :
✅ Fichier trouvé : jeliaudio/jeliaudio/griots_r1-45480-63500.wav
✅ Fichier trouvé : jeliaudio/jeliaudio/griots_r1-77180-79020.wav
✅ Fichier trouvé : jeliaudio/jeliaudio/griots_r1-111860-116180.wav


In [ ]:
# 🔁 Fusionner les deux
final_dataset = dataset_entries + jeli_entries

# 📊 Résumé
print(f"\n✅ Nombre total d'entrées finales : {len(final_dataset)}")
if final_dataset:
    print("🔍 Exemple d'entrée fusionnée :")
    print(final_dataset[0])



✅ Nombre total d'entrées finales : 1902
🔍 Exemple d'entrée fusionnée :
{'audio_filepath': 'audios/audios/Apprendre le Bambara Tranquillement au Lit  200 PhrasesLire ou Écouter ou Répéter  Zanga School_segment_911.wav', 'start': 0.0, 'end': 23.83, 'text': 'Tan in tila ka ɲɛ, nin sira in tila ka ɲɛ, nin ye mun fɛn ye?', 'translation': "Divisez correctement le dix, divisez correctement cette ligne, qu'est-ce que c'est ?"}


In [ ]:
import os
import torchaudio
from tqdm import tqdm

# Nettoyage des entrées dont les fichiers audio n'existent pas
final_dataset = [entry for entry in final_dataset if os.path.exists(entry["audio_filepath"])]

# Nouveau dossier pour les fichiers audio convertis
resampled_dir = "audios_resampled"
os.makedirs(resampled_dir, exist_ok=True)

resampled_entries = []

print("Rééchantillonnage des fichiers audio...")

for entry in tqdm(final_dataset):
    orig_path = entry["audio_filepath"]

    if not os.path.exists(orig_path):
        print(f"Fichier introuvable : {orig_path}")
        continue

    # Nouveau chemin
    filename = os.path.basename(orig_path)
    new_path = os.path.join(resampled_dir, filename)

    # Chargement + rééchantillonnage
    waveform, sample_rate = torchaudio.load(orig_path)

    if sample_rate != 16000:
        resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)
        waveform = resampler(waveform)

    torchaudio.save(new_path, waveform, 16000)

    # Mise à jour du chemin dans l'entrée
    resampled_entry = entry.copy()
    resampled_entry["audio_filepath"] = new_path
    resampled_entries.append(resampled_entry)

print(f"\nTous les fichiers ont été rééchantillonnés et enregistrés dans {resampled_dir}.")

# Suppression des fichiers audio orphelins dans le dossier "audios"
audio_dir = "audios/audios"
used_files = set(os.path.basename(entry["audio_filepath"]) for entry in final_dataset)

for file in os.listdir(audio_dir):
    if file not in used_files:
        full_path = os.path.join(audio_dir, file)
        os.remove(full_path)
        print(f"Fichier non utilisé supprimé : {full_path}")


Rééchantillonnage des fichiers audio...


100%|██████████| 1811/1811 [00:42<00:00, 42.71it/s]



Tous les fichiers ont été rééchantillonnés et enregistrés dans audios_resampled.
Fichier non utilisé supprimé : audios/audios/409.wav
Fichier non utilisé supprimé : audios/audios/242.mp3
Fichier non utilisé supprimé : audios/audios/342.wav
Fichier non utilisé supprimé : audios/audios/451.wav
Fichier non utilisé supprimé : audios/audios/283.wav
Fichier non utilisé supprimé : audios/audios/135.mp3
Fichier non utilisé supprimé : audios/audios/446.wav
Fichier non utilisé supprimé : audios/audios/242.wav
Fichier non utilisé supprimé : audios/audios/209.mp3
Fichier non utilisé supprimé : audios/audios/135.wav
Fichier non utilisé supprimé : audios/audios/219.mp3
Fichier non utilisé supprimé : audios/audios/444.wav
Fichier non utilisé supprimé : audios/audios/081.mp3
Fichier non utilisé supprimé : audios/audios/346.mp3
Fichier non utilisé supprimé : audios/audios/146.wav
Fichier non utilisé supprimé : audios/audios/126.wav
Fichier non utilisé supprimé : audios/audios/456.mp3
Fichier non utili

In [ ]:
from datasets import Dataset

# Crée un objet Dataset à partir de la liste de dictionnaires
hf_dataset = Dataset.from_list(resampled_entries)

# 🔍 Aperçu
hf_dataset = hf_dataset.shuffle(seed=42)
hf_dataset.select(range(1)).to_pandas()


,audio_filepath,start,end,text,translation
0,audios_resampled/griots_r22-559040-564320.wav,0.0,5.28,n'a ya fili a bɛ se ka cɛ tan ni kɔ la bin,


In [ ]:
from datasets import Audio

# On transforme le chemin en audio réel (auto-chargé à la volée)
hf_dataset = hf_dataset.cast_column("audio_filepath", Audio(sampling_rate=16000))

# Pour que ça soit plus clair, on renomme la colonne
hf_dataset = hf_dataset.rename_column("audio_filepath", "audio")

# Vérifions que tout est bon
hf_dataset[0]


{'audio': {'path': 'audios_resampled/griots_r22-559040-564320.wav',
  'array': array([ 6.10351562e-05,  6.10351562e-05,  1.83105469e-04, ...,
         -1.83105469e-04, -3.05175781e-05, -1.52587891e-04]),
  'sampling_rate': 16000},
 'start': 0.0,
 'end': 5.28,
 'text': "n'a ya fili a bɛ se ka cɛ tan ni kɔ la bin",
 'translation': ''}

In [ ]:
# pour la traduction
from datasets import DatasetDict

# Copie du dataset uniquement pour la traduction (on ne touche pas à hf_dataset)
translation_dataset = hf_dataset.remove_columns(
    [col for col in hf_dataset.column_names if col not in ["text", "translation"]]
)


In [ ]:
# Séparation aléatoire des données (90% train, 10% test)
translation_dataset = translation_dataset.train_test_split(test_size=0.1, seed=42)


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_checkpoint = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/3.55k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [ ]:
src_lang = "bm_Latn"
tgt_lang = "fra_Latn"
max_length = 128

def preprocess_translation(example):
    # On ajoute le token de langue cible (important pour NLLB !)
    example["input_ids"] = tokenizer(example["text"],
                                     truncation=True,
                                     padding="max_length",
                                     max_length=max_length,
                                     return_tensors="pt").input_ids[0]

    example["labels"] = tokenizer(example["translation"],
                                  truncation=True,
                                  padding="max_length",
                                  max_length=max_length,
                                  return_tensors="pt").input_ids[0]

    return example


In [ ]:
tokenized_dataset = translation_dataset.map(
    preprocess_translation,
    remove_columns=["text", "translation"],
    num_proc=1
)


Map:   0%|          | 0/49 [00:00<?, ? examples/s]

Map:   0%|          | 0/6 [00:00<?, ? examples/s]

In [ ]:
from transformers import Seq2SeqTrainingArguments


training_args = Seq2SeqTrainingArguments(
    output_dir="./results_nllb_large",
    evaluation_strategy="steps",
    eval_steps=1000,
    save_steps=1000,
    save_total_limit=3,
    learning_rate=5e-5,
    per_device_train_batch_size=8,  # augmente si possible
    per_device_eval_batch_size=8,
    num_train_epochs=10,  # tu peux ajuster après un 1er test
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir="./logs_nllb_large",
    logging_steps=50,
    fp16=True,
    predict_with_generate=True,
    report_to="none"
)





/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"  # Pour être doublement sûre 😄

from transformers import Seq2SeqTrainer, DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    callbacks=[]  # facultatif ici, mais garde-le si tu veux être tranquille
)


<ipython-input-17-eb280e99581a>:8: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [ ]:
trainer.train()


Epoch,Training Loss,Validation Loss
1,8.140300,8.681164
2,8.078800,8.142185
3,6.968800,7.763717
4,6.745800,7.543397
5,6.460100,7.427875


/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:3353: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 200}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


TrainOutput(global_step=65, training_loss=7.101705081646259, metrics={'train_runtime': 119.4177, 'train_samples_per_second': 2.052, 'train_steps_per_second': 0.544, 'total_flos': 66367578439680.0, 'train_loss': 7.101705081646259, 'epoch': 5.0})

In [ ]:
import evaluate
import numpy as np

metric = evaluate.load("sacrebleu")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    return metric.compute(predictions=decoded_preds, references=[[label] for label in decoded_labels])


In [ ]:
trainer.compute_metrics = compute_metrics
trainer.evaluate()


{'eval_loss': 7.427874565124512,
 'eval_score': 1.6498335211341666,
 'eval_counts': [51, 8, 1, 0],
 'eval_totals': [181, 175, 169, 163],
 'eval_precisions': [28.176795580110497,
  4.571428571428571,
  0.591715976331361,
  0.3067484662576687],
 'eval_bp': 0.7502919979062695,
 'eval_sys_len': 181,
 'eval_ref_len': 233,
 'eval_runtime': 6.0897,
 'eval_samples_per_second': 0.985,
 'eval_steps_per_second': 0.328,
 'epoch': 5.0}

In [ ]:
def prepare_dataset(example):
    audio = example["audio"]

    # Prétraitement avec le processor Whisper
    inputs = processor(
        audio=audio["array"],
        sampling_rate=audio["sampling_rate"],
        text=example["text"],
    )

    # Ajoute la longueur pour filtrage plus tard
    inputs["input_length"] = len(audio["array"]) / audio["sampling_rate"]

    return inputs


In [ ]:
hf_dataset = hf_dataset.map(
    prepare_dataset,
    remove_columns=hf_dataset.column_names,  # on garde que les colonnes utiles après transformation
    num_proc=1,
)


Map:   0%|          | 0/1811 [00:00<?, ? examples/s]

In [ ]:
max_input_length = 30.0

# hf_dataset = hf_dataset.filter(
#     lambda x: x["input_length"] < max_input_length,
#     input_columns=["input_length"],
# )
hf_dataset = hf_dataset.filter(
    lambda x: x < max_input_length,
    input_columns=["input_length"]
)


Filter:   0%|          | 0/1811 [00:00<?, ? examples/s]

In [ ]:
hf_dataset[0].keys()
# Tu verras normalement : ['input_features', 'labels', 'input_length']


dict_keys(['input_features', 'labels', 'input_length'])

In [ ]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import torch

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"][0]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch


In [ ]:
# 2. Activer le retour de l'attention mask
processor.feature_extractor.return_attention_mask = True
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token  # par défaut

In [ ]:
data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)


In [ ]:
from torch.utils.data import DataLoader

dataloader = DataLoader(hf_dataset, batch_size=2, collate_fn=data_collator)

batch = next(iter(dataloader))
batch.keys()  # devrait contenir 'input_features' et 'labels'


dict_keys(['input_features', 'attention_mask', 'labels'])

In [ ]:
import evaluate
from transformers.models.whisper.english_normalizer import BasicTextNormalizer

# Métrique d'évaluation : Word Error Rate (WER)
metric = evaluate.load("wer")

# Pour "nettoyer" les textes avant de calculer le WER
normalizer = BasicTextNormalizer()


In [ ]:
def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # Remplacer les -100 (tokens ignorés) par le token de padding
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    # Décodage brut (texte sans tokens spéciaux)
    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)

    # Word Error Rate sans normalisation
    wer_ortho = 100 * metric.compute(predictions=pred_str, references=label_str)

    # Word Error Rate avec normalisation
    pred_str_norm = [normalizer(pred) for pred in pred_str]
    label_str_norm = [normalizer(label) for label in label_str]

    # Supprimer les labels vides (problème courant)
    pred_str_norm = [p for i, p in enumerate(pred_str_norm) if len(label_str_norm[i]) > 0]
    label_str_norm = [l for l in label_str_norm if len(l) > 0]

    wer = 100 * metric.compute(predictions=pred_str_norm, references=label_str_norm)

    return {"wer_ortho": wer_ortho, "wer": wer}


In [ ]:
import torch
from transformers import WhisperForConditionalGeneration

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("📦 Modèle chargé sur :", device)

model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")

# 💡 Corrige le warning ici :
model.config.forced_decoder_ids = None

model.to(device)


📦 Modèle chargé sur : cuda


WhisperForConditionalGeneration(
  (model): WhisperModel(
    (encoder): WhisperEncoder(
      (conv1): Conv1d(80, 768, kernel_size=(3,), stride=(1,), padding=(1,))
      (conv2): Conv1d(768, 768, kernel_size=(3,), stride=(2,), padding=(1,))
      (embed_positions): Embedding(1500, 768)
      (layers): ModuleList(
        (0-11): 12 x WhisperEncoderLayer(
          (self_attn): WhisperSdpaAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=False)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
        

In [ ]:
from functools import partial

# Empêche une erreur lors de l'entraînement avec checkpointing
model.config.use_cache = False

# Fixer les options de génération
model.generate = partial(
    model.generate,
    language="french",  # 🔁 remplace ici par "french" si c'est ta langue cible
    task="transcribe",   # ou "transcribe" si c'est de la transcription directe
    use_cache=True
)


In [ ]:
import transformers
print(transformers.__version__)


4.51.3


In [ ]:
from transformers import Seq2SeqTrainingArguments
# pour mon petit dataset
training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-small-bambara",
    overwrite_output_dir=True,  # pour écraser les anciens runs
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=1e-5,
    lr_scheduler_type="constant_with_warmup",
    warmup_steps=10,
    num_train_epochs=10,  # remplacer max_steps
    gradient_checkpointing=False,  # facultatif
    fp16=True,
    fp16_full_eval=False,
    evaluation_strategy="epoch",  # mieux que "steps" pour petit dataset
    predict_with_generate=True,
    generation_max_length=225,
    save_strategy="epoch",
    save_total_limit=2,
    logging_steps=25,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=False,
)


/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
import transformers
print(transformers.__version__)


In [ ]:
from datasets import DatasetDict

# Diviser le dataset en 80% pour l'entraînement et 20% pour le test
hf_dataset_split = hf_dataset.train_test_split(test_size=0.2)

# Maintenant, hf_dataset_split contient "train" et "test"
train_dataset = hf_dataset_split["train"]
test_dataset = hf_dataset_split["test"]


In [ ]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_dataset,  # Ensemble d'entraînement
    eval_dataset=test_dataset,  # Ensemble de test
    data_collator=data_collator,  # Le collateur de données
    compute_metrics=compute_metrics,  # Calcul des métriques (par exemple, WER)
    tokenizer=processor,  # Le processor Whisper
)


<ipython-input-38-c3075a680a57>:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Wer Ortho,Wer
1,No log,2.780564,363.750000,359.248555
2,No log,1.589950,179.375000,230.635838
3,2.265000,1.196536,141.562500,144.508671
4,2.265000,1.027056,111.875000,107.225434
5,0.677900,0.972821,105.625000,102.601156
6,0.677900,0.963565,105.312500,101.156069
7,0.677900,0.978302,109.687500,106.358382
8,0.193100,0.963333,102.812500,99.710983
9,0.193100,0.966830,111.250000,105.202312
10,0.066900,0.958066,110.000000,104.624277


/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:3353: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 448, 'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 359, 503, 522, 542, 873, 893, 902, 918, 922, 931, 1350, 1853, 1982, 2460, 2627, 3246, 3253, 3268, 3536, 3846, 3961, 4183, 4667, 6585, 6647, 7273, 9061, 9383, 10428, 10929, 11938, 12033, 12331, 12562, 13793, 14157, 14635, 15265, 15618, 16553, 16604, 18362, 18956, 20075, 21675, 22520, 26130, 26161, 26435, 28279, 29464, 31650, 32302, 32470, 36865, 42863, 47425, 49870, 50254, 50258, 50360, 50361, 50362], 'begin_suppress_tokens': [220, 50257]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


TrainOutput(global_step=100, training_loss=0.8006967449188233, metrics={'train_runtime': 476.434, 'train_samples_per_second': 0.819, 'train_steps_per_second': 0.21, 'total_flos': 1.125483061248e+17, 'train_loss': 0.8006967449188233, 'epoch': 10.0})

In [ ]:
# pour wav2vec
from datasets import Dataset, Audio

# Créer un objet Dataset HuggingFace à partir de ta liste (resampled_entries)
wav2vec_ds = Dataset.from_list(resampled_entries)

# Mélange les données pour que ce soit bien réparti
wav2vec_ds = wav2vec_ds.shuffle(seed=42)

# Convertit les chemins en vrais objets audio
wav2vec_ds = wav2vec_ds.cast_column("audio_filepath", Audio(sampling_rate=16000))

# Renomme la colonne pour correspondre aux attentes de Wav2Vec2
wav2vec_ds = wav2vec_ds.rename_column("audio_filepath", "audio")

# Vérification d'un exemple
wav2vec_ds[0]



{'audio': {'path': 'audios_resampled/letraduc16_part3.wav',
  'array': array([5.73730469e-03, 8.48388672e-03, 7.35473633e-03, ...,
         0.00000000e+00, 3.05175781e-05, 1.52587891e-04]),
  'sampling_rate': 16000},
 'start': 0.3,
 'end': 22.3,
 'text': "Wa sabanan fana ye min ye , i bɛ yɔrɔ min na f'i ka kɛ cogoya bɛɛ la i ka yelen ye k'i to dibi la o kɔrɔ k'o tɛna falen yɔrɔnin kelen, ni k'o tɛna falen yɔrɔnin kelen k'o bɛna wagati ta, wa k'o bɛna wagati min ta kuma dɔ la k'o kadi kuma dɔ la k'o mandi, nka f'i ka laɲini jɔnjɔn in dɔn, n'i b'a dɔn i kan ka taa yɔrɔ min na sira janya o ma teli k'i tɔrɔ.",
 'translation': "Et puis, il y a aussi autre chose : là où tu es, fais tout ce que tu peux pour voir la lumière dans l'obscurité. Cela signifie que les choses ne changeront pas du jour au lendemain. Si elles ne changent pas immédiatement, elles prendront du temps. Et durant ce temps, certaines choses seront bonnes, d'autres mauvaises. Mais si tu connais ton objectif, si tu sais où tu

In [ ]:
from transformers import Wav2Vec2CTCTokenizer

# Créer un fichier vocab.json contenant les caractères uniques du bambara
# (Exemple pour le bambara : lettres + symboles spéciaux)
vocab_chars = list(set(" ".join(wav2vec_ds["text"])))  # Extrait les caractères uniques
vocab = {v: k for k, v in enumerate(sorted(vocab_chars))}

# Sauvegarder le vocabulaire dans un fichier JSON
import json
with open("vocab.json", "w") as f:
    json.dump(vocab, f)

# Charger le tokenizer personnalisé
tokenizer = Wav2Vec2CTCTokenizer.from_pretrained(
    ".",  # Dossier courant où se trouve vocab.json
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|",
)

In [ ]:
from transformers import Wav2Vec2FeatureExtractor, Wav2Vec2Processor

feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True,
)

processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer,
)

In [ ]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import torch

@dataclass
class DataCollatorCTCWithPadding:
    processor: Any
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # audio
        input_values = [f["input_values"] for f in features]
        batch = self.processor.pad(
            {"input_values": input_values},
            padding=self.padding,
            return_tensors="pt"
        )

        # labels
        label_features = [f["labels"] for f in features]
        labels_batch = self.processor.pad(
            {"input_ids": label_features},
            padding=self.padding,
            return_tensors="pt"
        )

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels

        return batch

wav2vec_data_collator = DataCollatorCTCWithPadding(processor=wav2vec_processor)


In [ ]:
import evaluate

wer_metric = evaluate.load("wer")

def compute_wav2vec_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = torch.argmax(torch.tensor(pred_logits), dim=-1)

    # Décodage des prédictions
    pred_str = wav2vec_processor.batch_decode(pred_ids)

    # Décodage des références
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = wav2vec_processor.tokenizer.pad_token_id
    label_str = wav2vec_processor.batch_decode(label_ids, group_tokens=False)

    wer = 100 * wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}


In [ ]:
from transformers import Wav2Vec2ForCTC

wav2vec_model = Wav2Vec2ForCTC.from_pretrained(
    "facebook/wav2vec2-large-xlsr-53",
    ctc_loss_reduction="mean",
    pad_token_id=wav2vec_processor.tokenizer.pad_token_id
)


In [ ]:
wav2vec_split = wav2vec_ds.train_test_split(test_size=0.2)
train_wav2vec = wav2vec_split["train"]
test_wav2vec = wav2vec_split["test"]


In [ ]:
from transformers import TrainingArguments

wav2vec_training_args = TrainingArguments(
    output_dir="./wav2vec2-bambara",
    group_by_length=True,
    per_device_train_batch_size=16,
    evaluation_strategy="steps",
    num_train_epochs=20,
    fp16=True,
    save_steps=400,
    eval_steps=400,
    logging_steps=50,
    learning_rate=3e-4,
    warmup_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=False
)


In [ ]:
from transformers import Trainer

wav2vec_trainer = Trainer(
    model=wav2vec_model,
    args=wav2vec_training_args,
    train_dataset=train_wav2vec,
    eval_dataset=test_wav2vec,
    tokenizer=wav2vec_processor.feature_extractor,
    data_collator=wav2vec_data_collator,
    compute_metrics=compute_wav2vec_metrics,
)

wav2vec_trainer.train()


In [ ]:
import tensorflow as tf
tf.test.gpu_device_name()

'/device:GPU:0'

In [ ]:
# ✅ Même code que pour Whisper (chargement JSON + rééchantillonnage audio)
# Seul le dossier de sortie change pour Wav2Vec2
wav2vec_resampled_dir = "wav2vec_audios_resampled"
os.makedirs(wav2vec_resampled_dir, exist_ok=True)

wav2vec_resampled_entries = []

for entry in tqdm(dataset_entries):
    orig_path = entry["audio_filepath"]
    filename = os.path.basename(orig_path)
    new_path = os.path.join(wav2vec_resampled_dir, filename)

    # Rééchantillonnage à 16kHz (identique)
    waveform, sample_rate = torchaudio.load(orig_path)
    if sample_rate != 16000:
        resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)
        waveform = resampler(waveform)
    torchaudio.save(new_path, waveform, 16000)

    wav2vec_resampled_entries.append({**entry, "audio_filepath": new_path})

100%|██████████| 55/55 [00:09<00:00,  5.90it/s]


In [ ]:
from datasets import Dataset, Audio

# Convertir en Dataset HuggingFace
wav2vec_dataset = Dataset.from_list(wav2vec_resampled_entries)
wav2vec_dataset = wav2vec_dataset.shuffle(seed=42)

# Chargement des fichiers audio
wav2vec_dataset = wav2vec_dataset.cast_column("audio_filepath", Audio(sampling_rate=16000))
wav2vec_dataset = wav2vec_dataset.rename_column("audio_filepath", "audio")

# Exemple de sortie
print(wav2vec_dataset[0]["audio"]["array"].shape)  # Vérifier la forme du signal

(367360,)


In [ ]:
from transformers import Wav2Vec2CTCTokenizer

# Extraire tous les caractères uniques du texte Bambara
chars = list(set(" ".join(wav2vec_dataset["text"])))
vocab = {v: k for k, v in enumerate(sorted(chars))}

# Sauvegarder le vocabulaire
with open("wav2vec_vocab.json", "w") as f:
    json.dump(vocab, f)

# Charger le tokenizer
wav2vec_tokenizer = Wav2Vec2CTCTokenizer.from_pretrained(
    ".",  # Dossier courant
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|",
)

In [ ]:
from transformers import Wav2Vec2FeatureExtractor, Wav2Vec2Processor

wav2vec_feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16000,
    padding_value=0.0,
    do_normalize=True,
)

wav2vec_processor = Wav2Vec2Processor(
    feature_extractor=wav2vec_feature_extractor,
    tokenizer=wav2vec_tokenizer,
)

In [ ]:
from transformers import Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor, Wav2Vec2Processor
import json

# 1. Création du vocabulaire spécialisé Bambara
def create_vocab(dataset):
    all_text = " ".join(dataset["text"])
    vocab_chars = list(set(all_text))
    vocab = {v: k for k, v in enumerate(sorted(vocab_chars))}
    vocab["[UNK]"] = len(vocab)
    vocab["[PAD]"] = len(vocab)
    return vocab

vocab = create_vocab(wav2vec_dataset)
with open("vocab.json", "w") as f:
    json.dump(vocab, f)

# 2. Chargement du tokenizer avec le bon vocabulaire
tokenizer = Wav2Vec2CTCTokenizer.from_pretrained(
    ".",
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|",
)

# 3. Configuration du feature extractor
feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16000,
    padding_value=0.0,
    do_normalize=True,
)

processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer
)

# 4. Fonction de prétraitement corrigée
def prepare_wav2vec_batch(batch):
    audio = batch["audio"]

    # Traitement audio
    inputs = processor(
        audio["array"],
        sampling_rate=audio["sampling_rate"],
        padding="max_length",
        max_length=16000*30,  # 30 secondes
        truncation=True,
        return_tensors="pt",
    )

    # Encodage du texte
    labels = processor.tokenizer(
        batch["text"],
        padding="max_length",
        max_length=100,  # Longueur max du texte
        truncation=True,
        return_tensors="pt",
    ).input_ids[0]

    return {
        "input_values": inputs.input_values[0],
        "attention_mask": inputs.attention_mask[0],
        "labels": labels,
    }

# 5. Application avec gestion d'erreur
try:
    wav2vec_dataset = wav2vec_dataset.map(
        prepare_wav2vec_batch,
        remove_columns=wav2vec_dataset.column_names,
        num_proc=1  # Réduire à 1 pour le debug
    )
    print("✅ Prétraitement réussi !")
except Exception as e:
    print(f"❌ Erreur : {str(e)}")
    # Debug : afficher un exemple problématique
    print("Exemple problématique :")
    print(wav2vec_dataset[0])

Map:   0%|          | 0/55 [00:00<?, ? examples/s]

❌ Erreur : 
Exemple problématique :
{'audio': {'path': 'wav2vec_audios_resampled/letraduc16_part3.wav', 'array': array([5.73730469e-03, 8.48388672e-03, 7.35473633e-03, ...,
       0.00000000e+00, 3.05175781e-05, 1.52587891e-04]), 'sampling_rate': 16000}, 'start': 0.3, 'end': 22.3, 'text': "Wa sabanan fana ye min ye , i bɛ yɔrɔ min na f'i ka kɛ cogoya bɛɛ la i ka yelen ye k'i to dibi la o kɔrɔ k'o tɛna falen yɔrɔnin kelen, ni k'o tɛna falen yɔrɔnin kelen k'o bɛna wagati ta, wa k'o bɛna wagati min ta kuma dɔ la k'o kadi kuma dɔ la k'o mandi, nka f'i ka laɲini jɔnjɔn in dɔn, n'i b'a dɔn i kan ka taa yɔrɔ min na sira janya o ma teli k'i tɔrɔ.", 'translation': "Et puis, il y a aussi autre chose : là où tu es, fais tout ce que tu peux pour voir la lumière dans l'obscurité. Cela signifie que les choses ne changeront pas du jour au lendemain. Si elles ne changent pas immédiatement, elles prendront du temps. Et durant ce temps, certaines choses seront bonnes, d'autres mauvaises. Mais si tu conn

In [ ]:
import torch
from transformers import (
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor,
    Wav2Vec2ForCTC,
    TrainingArguments,
    Trainer
)
from datasets import Dataset, Audio
import json
import evaluate
import numpy as np

# Vérification GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️  Device utilisé : {device}")

🖥️  Device utilisé : cuda


In [ ]:
# Création du vocabulaire depuis vos données
def create_vocab(dataset):
    all_text = " ".join(dataset["text"])
    vocab_chars = sorted(list(set(all_text)))
    vocab = {v: k for k, v in enumerate(vocab_chars)}
    vocab["[UNK]"] = len(vocab)
    vocab["[PAD]"] = len(vocab)
    return vocab

vocab = create_vocab(hf_dataset)
with open("vocab.json", "w") as f:
    json.dump(vocab, f)

# Tokenizer personnalisé
tokenizer = Wav2Vec2CTCTokenizer.from_pretrained(
    ".",
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|",
)

In [ ]:
feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16000,
    padding_value=0.0,
    do_normalize=True,
)

processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer
)

In [ ]:
def prepare_batch(batch):
    audio = batch["audio"]

    # 1. Traitement audio (séparé)
    audio_inputs = processor.feature_extractor(
        raw_speech=audio["array"],
        sampling_rate=audio["sampling_rate"],
        padding="max_length",
        max_length=16000*30,
        truncation=True,
        return_tensors="pt",
        return_attention_mask=True
    )

    # 2. Traitement texte (séparé)
    text_inputs = processor.tokenizer(
        batch["text"],
        padding="max_length",
        max_length=100,
        truncation=True,
        return_tensors="pt"
    )

    return {
        "input_values": audio_inputs["input_values"][0],
        "attention_mask": audio_inputs["attention_mask"][0],
        "labels": text_inputs["input_ids"][0],
    }

def safe_prepare_batch(batch):
    try:
        # Vérifications de base
        if not isinstance(batch["audio"]["array"], np.ndarray):
            raise ValueError("Audio doit être un numpy array")
        if len(batch["audio"]["array"]) == 0:
            raise ValueError("Audio vide")
        if not isinstance(batch["text"], str):
            raise ValueError("Texte doit être une string")

        return prepare_batch(batch)
    except Exception as e:
        print(f"\n❌ Erreur sur l'exemple:")
        print(f"Texte: {batch.get('text', 'N/A')[:50]}...")
        print(f"Type audio: {type(batch['audio'].get('array', 'N/A'))}")
        print(f"Taille audio: {len(batch['audio'].get('array', []))}")
        print(f"Sample rate: {batch['audio'].get('sampling_rate', 'N/A')}")
        raise RuntimeError(f"Erreur de prétraitement: {str(e)}") from e

# Test sur un seul échantillon d'abord
try:
    print("🔍 Test sur un échantillon...")
    test_sample = prepare_batch(hf_dataset[0])
    print("✅ Test réussi. Shapes:")
    print(f"input_values: {test_sample['input_values'].shape}")
    print(f"attention_mask: {test_sample['attention_mask'].shape}")
    print(f"labels: {test_sample['labels'].shape}")

    # Si le test passe, traiter tout le dataset
    print("\n🚀 Traitement complet du dataset...")
    encoded_dataset = hf_dataset.map(
        safe_prepare_batch,
        remove_columns=hf_dataset.column_names,
        num_proc=1  # Commencer avec 1 pour debug
    )
    print("🌈 Prétraitement terminé avec succès !")
except Exception as e:
    print(f"❌ Échec du prétraitement: {str(e)}")
    print("Conseils de dépannage:")
    print("1. Vérifiez que tous les audios sont au format numpy array")
    print("2. Assurez-vous qu'aucun texte n'est None ou vide")
    print(f"3. Vérifiez le vocab.json dans {os.getcwd()}")
    if 'processor' in locals():
        print(f"4. Taille du vocabulaire: {len(processor.tokenizer)}")

🔍 Test sur un échantillon...
✅ Test réussi. Shapes:
input_values: torch.Size([480000])
attention_mask: torch.Size([480000])
labels: torch.Size([100])

🚀 Traitement complet du dataset...


Map:   0%|          | 0/55 [00:00<?, ? examples/s]

🌈 Prétraitement terminé avec succès !


In [ ]:
@dataclass
class DataCollatorCTC:
    processor: Wav2Vec2Processor
    padding: bool = True

    def __call__(self, features):
        input_features = [{"input_values": f["input_values"]} for f in features]
        batch = self.processor.pad(input_features, padding="longest", return_tensors="pt")

        labels = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.pad(labels, return_tensors="pt")

        batch["labels"] = labels_batch["input_ids"]
        return batch

data_collator = DataCollatorCTC(processor=processor)

wer_metric = evaluate.load("wer")
def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)

    pred_str = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(pred.label_ids, group_tokens=False)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

In [ ]:
model = Wav2Vec2ForCTC.from_pretrained(
    "facebook/wav2vec2-large-xlsr-53",
    attention_dropout=0.1,
    hidden_dropout=0.1,
    feat_proj_dropout=0.1,
    ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),
).to(device)

pytorch_model.bin:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-large-xlsr-53 and are newly initialized: ['lm_head.bias', 'lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./wav2vec2-bambara",
    overwrite_output_dir=True,
    per_device_train_batch_size=2,  # Réduit pour gérer les longs audios
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=2,  # Compensation du petit batch_size
    learning_rate=3e-4,
    lr_scheduler_type="constant_with_warmup",
    warmup_steps=10,
    num_train_epochs=10,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    logging_steps=25,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=False,
    gradient_checkpointing=True,  # Réduit l'utilisation mémoire
    optim="adafactor",  # Optimiseur moins gourmand en mémoire
)

# Initialisation du Trainer avec gestion mémoire
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset_split["train"],
    eval_dataset=dataset_split["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
# Ajoutez ceci avant l'entraînement pour optimiser la mémoire
import torch
torch.cuda.empty_cache()

# Et cette option pour réduire la fragmentation mémoire
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"

In [ ]:
# Callback pour des logs similaires à Whisper
from transformers import TrainerCallback

class BambaraMetricsCallback(TrainerCallback):
    def on_evaluate(self, args, state, control, **kwargs):
        metrics = state.log_history[-1]
        print(f"\nEpoch {state.epoch} - WER: {metrics.get('eval_wer', 'N/A'):.2f}%")

trainer.add_callback(BambaraMetricsCallback())

In [ ]:
print("🚀 Démarrage de l'entraînement sur 10 epochs...")
try:
    trainer.train()
    print("✅ Entraînement terminé avec succès!")

    # # Sauvegarde finale
    # trainer.save_model("./wav2vec2-bambara-final")
    # print("Modèle sauvegardé pour évaluation.")

except Exception as e:
    print(f"❌ Erreur: {str(e)}")
    print("Solutions possibles:")
    print("1. Réduire encore le batch_size à 1")
    print("2. Diminuer la longueur maximale des audios")
    print("3. Utiliser gradient_checkpointing=True")

🚀 Démarrage de l'entraînement sur 10 epochs...


Epoch,Training Loss,Validation Loss,Wer
1,No log,53.553261,1.000000
2,No log,44.966660,1.000000
3,49.003900,5.636307,1.000000



Epoch 1.0 - WER: 1.00%

Epoch 2.0 - WER: 1.00%

Epoch 3.0 - WER: 1.00%


KeyboardInterrupt: 

In [ ]:
# Utilisez la même fonction compute_metrics que pour Whisper
def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)

    pred_str = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(pred.label_ids, group_tokens=False)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

In [ ]:
training_args.eval_strategy = "epoch"  # Comme votre config Whisper
training_args.save_strategy = "epoch"

In [ ]:
from collections import Counter
import json
import os

def extract_vocab(dataset):
    all_texts = " ".join(dataset["text"])
    vocab = list(set(all_texts))
    vocab_dict = {char: idx for idx, char in enumerate(sorted(vocab))}
    vocab_dict["|"] = vocab_dict[" "]
    del vocab_dict[" "]
    vocab_dict["[UNK]"] = len(vocab_dict)
    vocab_dict["[PAD]"] = len(vocab_dict)
    return vocab_dict

vocab_dict = extract_vocab(hf_dataset)
print(f"🧠 Vocab size: {len(vocab_dict)}")


🧠 Vocab size: 54


In [ ]:
with open("vocab_bambara.json", "w", encoding="utf-8") as vocab_file:
    json.dump(vocab_dict, vocab_file, ensure_ascii=False)


In [ ]:
from transformers import Wav2Vec2CTCTokenizer

tokenizer = Wav2Vec2CTCTokenizer(
    "vocab_bambara.json",
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|"
)
tokenizer.save_pretrained("bambara_tokenizer")


('bambara_tokenizer/tokenizer_config.json',
 'bambara_tokenizer/special_tokens_map.json',
 'bambara_tokenizer/vocab.json',
 'bambara_tokenizer/added_tokens.json')

In [ ]:
from transformers import Wav2Vec2Processor, Wav2Vec2FeatureExtractor

feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True
)

processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)
processor.save_pretrained("bambara_processor")


[]

In [ ]:
def prepare_dataset_wav2vec(example):
    audio = example["audio"]
    inputs = processor(audio["array"], sampling_rate=audio["sampling_rate"])
    with processor.as_target_processor():
        labels = processor(example["text"]).input_ids

    inputs["labels"] = labels
    return inputs

wav2vec_dataset = hf_dataset.map(
    prepare_dataset_wav2vec,
    remove_columns=hf_dataset.column_names,
    num_proc=1
)


Map:   0%|          | 0/55 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/transformers/models/wav2vec2/processing_wav2vec2.py:174: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your audio inputs, or in a separate call.
  warnings.warn(


In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Union
import torch

@dataclass
class MyDataCollatorCTCWithPadding:
    processor: any
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Regroupe les audios
        input_features = [{"input_values": f["input_values"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        batch = self.processor.feature_extractor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt"
        )
        labels_batch = self.processor.tokenizer.pad(
            label_features,
            padding=self.padding,
            return_tensors="pt"
        )

        # Important pour la perte CTC : masquer les pads
        labels = labels_batch["input_ids"].masked_fill(labels_batch["input_ids"] == self.processor.tokenizer.pad_token_id, -100)
        batch["labels"] = labels

        return batch


In [ ]:
from transformers import Wav2Vec2ForCTC

model = Wav2Vec2ForCTC.from_pretrained(
    "facebook/wav2vec2-large-xlsr-53",
    vocab_size=len(processor.tokenizer),
    pad_token_id=processor.tokenizer.pad_token_id
)


Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-large-xlsr-53 and are newly initialized: ['lm_head.bias', 'lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
